# Todo #6 — 特征工程（独立 Notebook）

> 从 [`credit-fraud-project-work.ipynb`](credit-fraud-project-work.ipynb) 拆出，专注 **Family A 门控交叉** 与 **OOF Autoencoder 异常分**。

**目标：** 降 FP（1 EUR 误报带优先）> 降 FN；训练期用 **AUC-PR** ablation 筛选（**基线 = BASE_FEATURES**），通过特征工程定稿 `MODEL_FEATURES_V2`。

**本节结构：**
1. 环境与数据（镜像主 notebook）
2. 建模工具（镜像 2.0）
3. Family A：`is_one_euro × Top-V`
4. Family B：占位（待做）
5. OOF Autoencoder 重构误差
6. 定稿 `MODEL_FEATURES_V2`


## 0. 环境与数据加载


In [1]:
# --- 0. 环境与数据加载 ---
# 镜像主 notebook；读入 creditcard.csv 并记录全局欺诈率。
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)



def resolve_creditcard_path() -> Path:
    """自 cwd 向上查找项目根下的 input/creditcard.csv（兼容 src/ 与 src/feature-engineering/）。"""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'input' / 'creditcard.csv'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        '未找到 input/creditcard.csv；请在项目根或 src/feature-engineering 下运行 notebook。'
    )


DATA_PATH = resolve_creditcard_path()


def read_creditcard_csv(path) -> pd.DataFrame:
    """依次尝试 utf-8 / 容错 utf-8 / latin-1，避免 UnicodeDecodeError。"""
    for kwargs in (
        {'encoding': 'utf-8'},
        {'encoding': 'utf-8', 'encoding_errors': 'replace'},
        {'encoding': 'latin-1'},
    ):
        try:
            return pd.read_csv(path, **kwargs)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1, 'failed to decode creditcard.csv')


df = read_creditcard_csv(DATA_PATH)
FRAUD_RATE = df['Class'].mean()
print(f'行数: {len(df):,}  |  欺诈笔数: {df["Class"].sum()}  |  欺诈率: {FRAUD_RATE:.4f}')


行数: 284,807  |  欺诈笔数: 492  |  欺诈率: 0.0017


## 1. 建模工具（镜像主 notebook 2.0）


In [2]:
# --- 2.0 1.7 特征构造 + 可复用验证工具（精简镜像）---
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
)
import lightgbm as lgb
import xgboost as xgb

V_COLS = [c for c in df.columns if c.startswith('V')]
BASE_FEATURES = V_COLS + ['Amount', 'Time']

EDA_FEATURES = [
    'log1p_amount',
    'hours_since_start',
    'is_micro_testing',
    'is_small_testing',
    'is_one_euro',
    'is_inactive',
]

EARLY_STOPPING_ROUNDS = 50
MAX_BOOST_ROUNDS = 1500
ES_FRAC = 0.20  # 早停监控集占当前 train 折比例（80% fit / 20% ES；比 25% 多留训练数据）
DEFAULT_CLASSIFICATION_THRESHOLD = 0.5
TOP_V_K = 6  # Family A/B 门控交叉用的 Top-V 个数
CV_N_SPLITS = 5           # 组合矩阵 / ablation 可选 CV 折数
CV_RANDOM_STATE = 42


def build_eda_features(data: pd.DataFrame) -> pd.DataFrame:
    """构造 1.7 衍生特征。"""
    out = data.copy()
    out['log1p_amount'] = np.log1p(out['Amount'])
    out['hours_since_start'] = (out['Time'] // 3600).astype(int)
    out['is_micro_testing'] = out['Amount'] < 1
    out['is_small_testing'] = (out['Amount'] > 1) & (out['Amount'] <= 5)  # is_small_testing换成(1,5]试试
    out['is_one_euro'] = out['Amount'] == 1.0
    hourly_cnt = out.groupby('hours_since_start').size()
    inactive_hours = hourly_cnt[hourly_cnt < hourly_cnt.median()].index
    out['is_inactive'] = out['hours_since_start'].isin(inactive_hours)
    return out


def _split_early_stop_set(X_tr, y_tr, es_frac=ES_FRAC, random_state=42):
    """切分出用来验证早停的数据"""
    return train_test_split(
        X_tr, y_tr, test_size=es_frac, random_state=random_state, stratify=y_tr,
    )


def make_classifier(model_name: str, y_train: pd.Series, params: dict | None = None):
    """构造树（logloss + 类别不平衡权重）"""
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight='balanced', random_state=42, verbose=-1,
        )
        defaults.update(params)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=42, eval_metric='logloss', verbosity=0,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    return xgb.XGBClassifier(**defaults)


def fit_classifier(clf, model_name: str, X_tr, y_tr, X_es=None, y_es=None):
    """训练（logloss + 早停）"""
    if X_es is None or y_es is None:
        clf.fit(X_tr, y_tr)
        return clf
    if model_name == 'LightGBM':
        clf.fit(
            X_tr, y_tr, eval_set=[(X_es, y_es)], eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
    else:
        clf.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
    return clf


def eval_classifier(
    model_name: str, X_train, y_train, X_val, y_val,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD, random_state: int = 42,
) -> dict:
    """不交叉验证，直接训练评估模型"""
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    clf = make_classifier(model_name, y_fit)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
    proba = clf.predict_proba(X_val)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    return {
        '模型': model_name, '特征数': X_train.shape[1],
        'AUC-PR': average_precision_score(y_val, proba),
        f'F1@{threshold}': f1_score(y_val, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y_val, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y_val, pred, zero_division=0),
        'FP': fp, 'FN': fn, 'proba': proba, 'pred': pred, 'threshold': threshold,
    }


def cross_val_auc_pr(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> tuple[float, float, list[float]]:
    """StratifiedKFold：返回 (AUC-PR 均值, 标准差, 各折分数列表)。"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_scores: list[float] = []
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        fold_scores.append(float(average_precision_score(y_va, clf.predict_proba(X_va)[:, 1])))
    arr = np.array(fold_scores)
    return float(arr.mean()), float(arr.std(ddof=0)), fold_scores


def get_oof_proba(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> np.ndarray:
    """5-fold OOF 概率，用于 CV 下汇总 FP/FN（@threshold）。"""
    oof = np.zeros(len(y), dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        oof[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]
    return oof


def cross_val_eval(
    model_name: str,
    data: pd.DataFrame,
    feature_cols: list,
    y_col: str = 'Class',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """CV 评估：一轮 StratifiedKFold 同时算各折 AUC-PR 与 OOF 概率（避免重复训练）。"""
    X = data[feature_cols]
    y = data[y_col]
    oof_proba = np.zeros(len(y), dtype=float)
    fold_scores: list[float] = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        proba_va = clf.predict_proba(X_va)[:, 1]
        fold_scores.append(float(average_precision_score(y_va, proba_va)))
        oof_proba[va_idx] = proba_va

    arr = np.array(fold_scores)
    pred = (oof_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        '模型': model_name,
        '特征数': len(feature_cols),
        'AUC-PR_mean': float(arr.mean()),
        'AUC-PR_std': float(arr.std(ddof=0)),
        'fold_AUC-PR': fold_scores,
        f'F1@{threshold}': f1_score(y, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y, pred, zero_division=0),
        'FP': int(fp),
        'FN': int(fn),
    }


def feature_ablation(
    data: pd.DataFrame, candidate_features=None, base_features=None,
    model_name: str = 'LightGBM', test_size: float = 0.2, random_state: int = 42,
    include_all_combo: bool = True,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> pd.DataFrame:
    """特征消融；并列输出 FP / FN（@threshold）。"""
    if base_features is None:
        base_features = BASE_FEATURES
    candidate_features = list(candidate_features or [])
    y = data['Class']
    X_train, X_val, y_train, y_val = train_test_split(
        data[base_features], y, test_size=test_size, random_state=random_state, stratify=y,
    )
    val_idx = y_val.index
    specs = [('基线', base_features)]
    for f in candidate_features:
        specs.append((f'+{f}', base_features + [f]))
    if include_all_combo and candidate_features:
        specs.append(('+全部候选', base_features + candidate_features))
    rows, base_ap = [], None
    for label, cols in specs:
        missing = [c for c in cols if c not in data.columns]
        if missing:
            raise KeyError(f'缺少列: {missing}')
        res = eval_classifier(
            model_name, data.loc[X_train.index, cols], y_train,
            data.loc[val_idx, cols], y_val, threshold=threshold,
        )
        if base_ap is None:
            base_ap = res['AUC-PR']
        rows.append({
            '特征集': label, '特征数': res['特征数'], 'AUC-PR': res['AUC-PR'],
            f"F1@{threshold}": res[f'F1@{threshold}'],
            f"Precision@{threshold}": res[f'Precision@{threshold}'],
            f"Recall@{threshold}": res[f'Recall@{threshold}'],
            'FP': res['FP'], 'FN': res['FN'],
            'Δ AUC-PR': res['AUC-PR'] - base_ap,
        })
    return pd.DataFrame(rows).sort_values('AUC-PR', ascending=False).reset_index(drop=True)


df_model = build_eda_features(df)
print('衍生特征已构造；ablation 基线 BASE_FEATURES：')
print(BASE_FEATURES)


衍生特征已构造；ablation 基线 BASE_FEATURES：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time']


## 6.0 基础设施：Top-V 选择、门控交叉、OOF Autoencoder


In [3]:
# =============================================================================
# 6.0 特征工程基础设施
# -----------------------------------------------------------------------------
# 本 cell 提供三类能力：
#   1. pick_top_v_features  — 用树模型 importance 选出与欺诈最相关的 Top-K 个 V 列
#   2. build_cross_features — 构造「？ × V」交叉特征（Family A/B 共用）
#   3. oof_ae_reconstruction_error — OOF Autoencoder，产出无泄露的异常分 ae_oof_error
# =============================================================================

# --- 依赖导入 ---
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

# --- Autoencoder 超参数（可按训练时间 / 效果微调）---
AE_INPUT_COLS = V_COLS          # AE 只吃 V1–V28（PCA 分量），不含 Amount/Time（Amount 强偏态）
AE_LATENT_DIM = 8               # 瓶颈层维度：压缩正常流形，欺诈/异常更难重构
AE_MAX_EPOCHS = 40              # 每折最大 epoch；实际由 EarlyStopping 提前结束
AE_BATCH_SIZE = 256             # mini-batch 大小
AE_PATIENCE = 5                 # val_loss 连续 5 epoch 不降则停训
AE_MAX_NORMAL_SAMPLES = 50_000  # 每折 normal 子采样上限（全表 normal≈28 万，全训太慢）
AE_RANDOM_STATE = 42            # 控制 KFold 划分、子采样、TF 随机种子


def pick_top_v_features(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    k: int = TOP_V_K,
    model_name: str = 'LightGBM',
    random_state: int = 42,
) -> list:
    """
    在指定特征集上训一版树分类器，按 feature_importances_（gain）取 Top-K 个 V 列。

      - feature_cols: list | None  → Python 3.10+ 联合类型；None 时用 BASE_FEATURES

    为何需要：Family A 交叉项是 is_one_euro × V_i，试图解决FP误报，不能对 28 个 V 全交叉（维数爆炸），
    先用模型 importance 筛出与欺诈最相关的 V，再与 is_one_euro 相乘。
    """
    feature_cols = feature_cols or BASE_FEATURES  # `or`：None / 空列表时回退默认
    y = data['Class']
    X = data[feature_cols]

    # stratify=y：分层抽样，保证 train/val 欺诈率与全表一致（极不平衡数据必做）
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )

    clf = make_classifier(model_name, y_train)
    # 再从 X_train 切 ES_FRAC（20%）作早停监控集（与 2.0 eval_classifier 一致）
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)

    # feature_importances_ 与 feature_cols 一一对应；只保留 V 列并降序
    imp = pd.Series(clf.feature_importances_, index=feature_cols)
    v_imp = imp[V_COLS].sort_values(ascending=False)
    top_v = list(v_imp.head(k).index)  # .head(k) 取前 k 个 index，转 list
    print(f'Top-{k} V（{model_name} gain）：{top_v}')
    return top_v


def build_cross_features(
    data: pd.DataFrame,
    top_v: list,
    gate_col: str = 'is_one_euro',
    prefix: str = 'one_euro',
) -> tuple[pd.DataFrame, list]:
    """
    特征交叉：new_col = gate_col × V_i。

      - 返回 tuple[pd.DataFrame, list]：新 DataFrame + 新列名列表（供 ablation 传入）

    业务含义：one_euro_Vi 等同于告诉树：「这一维 Vi 的信息，请只在 1 EUR 子空间里用。」
    帮助树模型区分「1 EUR 正常探测」与「1 EUR 欺诈」在 V 空间上的差异（降 FP）。
    否则一棵树更可能在全表上学：Vi > 某阈值 → 更像欺诈
    这个规则会作用在 所有金额 上，包括大量非 1 EUR 交易
    在 1 EUR 上，正常探测和欺诈的 Vi 可能重叠很大，全局阈值就容易误伤正常 1 EUR 交易（FP）
    """
    out = data.copy()            
    gate = out[gate_col].astype(float)  # bool → 0.0/1.0，便于与连续 V 相乘
    new_cols = []
    for v in top_v:
        name = f'{prefix}_{v}'  # 如 one_euro_V14
        out[name] = gate * out[v]   # 逐元素相乘（Series × Series）
        new_cols.append(name)
    return out, new_cols


def _build_ae(input_dim: int, latent_dim: int = AE_LATENT_DIM) -> keras.Model:
    """
    全连接 Autoencoder：input_dim → 16 → latent → 16 → input_dim。

    Keras Functional API：
      - keras.Input(shape=(input_dim,))：定义输入张量形状 (batch, input_dim)
      - 每层 Dense(...)(x)：函数式 API，x 是上一层的输出
      - 最后一层 Dense(input_dim) 无激活：重构原始维度，配合 MSE loss
      - keras.Model(inp, out)：指定输入/输出节点，构成可训练模型

    中间层 ReLU；瓶颈 latent 迫使模型学正常样本的低维表示（？）。
    """
    inp = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(16, activation='relu')(inp)
    x = keras.layers.Dense(latent_dim, activation='relu')(x)
    x = keras.layers.Dense(16, activation='relu')(x)
    out = keras.layers.Dense(input_dim)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
    return model


def oof_ae_reconstruction_error(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    y_col: str = 'Class',
    n_splits: int = 5,
    random_state: int = AE_RANDOM_STATE,
) -> np.ndarray:
    """
    Out-of-Fold Autoencoder 重构误差（样本级 MSE，越大越像异常）。

    核心防泄露设计：
      - 每折 AE 只在 train 折的 **正常样本** 上训练（学「正常流形」）
      - 重构误差只算 **val 折** 样本 → 该样本从未参与该折 AE 的训练
      - 禁止用 in-sample 误差当特征（否则欺诈/正常在训练集上的误差会偏乐观）

    返回值：长度 = len(data) 的 np.ndarray，与 data 行序对齐，可直接 df['ae_oof_error'] = ...
    """
    feature_cols = feature_cols or AE_INPUT_COLS
    X = data[feature_cols].values.astype(np.float32) 
    y = data[y_col].values
    oof_err = np.zeros(len(y), dtype=np.float32)      # 预分配空间，按 va_idx 填入各折结果

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        normal_tr = tr_idx[y[tr_idx] == 0]            # 训练 AE 时剔除欺诈，只学 normal

        # 子采样：normal 过多时随机抽 AE_MAX_NORMAL_SAMPLES 条，加速且通常足够
        if len(normal_tr) > AE_MAX_NORMAL_SAMPLES:
            rng = np.random.default_rng(random_state + fold)  # 每折不同种子，可复现
            normal_tr = rng.choice(normal_tr, size=AE_MAX_NORMAL_SAMPLES, replace=False)

        X_normal = X[normal_tr]
        scaler = StandardScaler()
        X_normal_s = scaler.fit_transform(X_normal)     # 只在 normal 上 fit μ、σ
        X_va_s = scaler.transform(X[va_idx])          # val 用同一 scaler transform（含欺诈）

        tf.keras.backend.clear_session()                # 每折新建模型前清图，防内存累积
        tf.random.set_seed(random_state + fold)
        ae = _build_ae(X.shape[1])

        es = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=AE_PATIENCE, restore_best_weights=True, verbose=0,
        )
        # 从 normal 训练集末尾切 10% 作 AE 内部 val（仅用于早停，不是 OOF val）
        n_val = max(1, int(0.1 * len(X_normal_s)))
        ae.fit(
            X_normal_s[:-n_val], X_normal_s[:-n_val],   # 输入=目标（自编码器）
            validation_data=(X_normal_s[-n_val:], X_normal_s[-n_val:]),
            epochs=AE_MAX_EPOCHS, batch_size=AE_BATCH_SIZE,
            callbacks=[es], verbose=0,
        )

        recon = ae.predict(X_va_s, verbose=0)           # shape: (n_val_fold, n_features)
        # axis=1：对每个样本的所有特征维求 MSE 均值 → 标量异常分
        oof_err[va_idx] = np.mean((X_va_s - recon) ** 2, axis=1)
        print(f'  fold {fold}/{n_splits} 完成（normal 训练样本 {len(normal_tr):,}）')
    return oof_err


# --- 执行：在 BASE 特征上跑一版 LGB，得到 Family A 要交叉的 Top-6 V ---
TOP_V = pick_top_v_features(df_model, BASE_FEATURES, k=TOP_V_K, model_name='LightGBM')


Top-6 V（LightGBM gain）：['V14', 'V4', 'V7', 'V16', 'V12', 'V26']


## 6.1 Family A — 1 EUR 特征交叉（FP 优先）

`one_euro_{Vi} = is_one_euro × V_i`，仅在 1 EUR 区激活对应主成分信号。


In [4]:
from IPython.display import display

# =============================================================================
# 6.1 Family A — 1 EUR 门控交叉 + ablation
# -----------------------------------------------------------------------------
# 流程：构造交叉列 → 对 LightGBM / XGBoost 分别做 feature_ablation
# ablation 基线默认为 BASE_FEATURES（V + Amount + Time），见 §1 的 feature_ablation
# =============================================================================

# build_cross_features 返回：
#   df_fe          — 在 df_model 上追加 one_euro_V* 列后的完整表
#   CROSS_FAMILY_A — 新列名列表，如 ['one_euro_V14', 'one_euro_V4', ...]
df_fe, CROSS_FAMILY_A = build_cross_features(
    df_model, TOP_V, gate_col='is_one_euro', prefix='one_euro',
)

print(f'Family A 共 {len(CROSS_FAMILY_A)} 列：')
# 展示「列名 → 定义」对照表，便于报告引用
display(pd.DataFrame({'特征': CROSS_FAMILY_A, '定义': [f'is_one_euro × {v}' for v in TOP_V]}))

# --- LightGBM ablation ---
# include_all_combo=True：除「基线」「+单列」外，再跑一行「+全部候选」（6 列一起加）
# 6.4 定稿用「+全部候选」行的 Δ AUC-PR 判断是否整族入选
print('\n=== Family A ablation（LightGBM，基线=BASE_FEATURES）===')
family_a_lgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='LightGBM', include_all_combo=True,
)
display(family_a_lgb.round(4))  # round(4) 便于阅读；Δ AUC-PR 相对基线行

# --- XGBoost ablation（同一划分、同一候选，对比两模型是否一致受益）---
print('\n=== Family A ablation（XGBoost）===')
family_a_xgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='XGBoost', include_all_combo=True,
)
display(family_a_xgb.round(4))


Family A 共 6 列：


,特征,定义
0,one_euro_V14,is_one_euro × V14
1,one_euro_V4,is_one_euro × V4
2,one_euro_V7,is_one_euro × V7
3,one_euro_V16,is_one_euro × V16
4,one_euro_V12,is_one_euro × V12
5,one_euro_V26,is_one_euro × V26



=== Family A ablation（LightGBM，基线=BASE_FEATURES）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+one_euro_V26,31,0.8801,0.8513,0.8557,0.8469,14,15,0.0034
1,+全部候选,36,0.8775,0.8469,0.8469,0.8469,15,15,0.0008
2,+one_euro_V14,31,0.8772,0.8691,0.8925,0.8469,10,15,0.0005
3,+one_euro_V16,31,0.8769,0.8513,0.8557,0.8469,14,15,0.0002
4,基线,30,0.8767,0.8469,0.8469,0.8469,15,15,0.0000
5,+one_euro_V4,31,0.8760,0.8586,0.8817,0.8367,11,16,-0.0006
6,+one_euro_V12,31,0.8750,0.8646,0.8830,0.8469,11,15,-0.0016
7,+one_euro_V7,31,0.8739,0.8513,0.8557,0.8469,14,15,-0.0028



=== Family A ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+one_euro_V16,31,0.8816,0.8617,0.9000,0.8265,9,17,0.0060
1,+one_euro_V12,31,0.8807,0.8617,0.9000,0.8265,9,17,0.0051
2,+one_euro_V14,31,0.8790,0.8601,0.8737,0.8469,12,15,0.0034
3,+one_euro_V26,31,0.8782,0.8632,0.8913,0.8367,10,16,0.0026
4,+one_euro_V4,31,0.8774,0.8677,0.9011,0.8367,9,16,0.0018
5,基线,30,0.8756,0.8586,0.8817,0.8367,11,16,0.0000
6,+one_euro_V7,31,0.8715,0.8497,0.8632,0.8367,13,16,-0.0041
7,+全部候选,36,0.8703,0.8617,0.9000,0.8265,9,17,-0.0052


## 6.2 Family B（占位）+ Family C 说明

**Family B（待做）：** `small_{Vi} = is_small_testing × V_i`，针对 1–100 EUR 漏报带（FN 次要目标）。
镜像 6.1，将 `gate_col='is_small_testing'`、`prefix='small'` 即可。

**Family C（本轮不做）：** 可选 `log1p_amount × is_inactive`、`log1p_amount × V_top` 等可解释连续门控。


In [5]:
# =============================================================================
# 6.2 Family B — 占位（尚未实现）
# -----------------------------------------------------------------------------
# 设计：与 6.1 镜像，门控改为 is_small_testing（1–20 EUR 小额带），针对 FN 次要目标。
# 取消注释下方代码即可跑通；TOP_V 可复用 6.0 结果，也可在 BASE 上重新 pick_top_v。
# =============================================================================

# TODO: 待做 — 镜像 6.1
# df_fe, CROSS_FAMILY_B = build_cross_features(
#     df_fe, TOP_V, gate_col='is_small_testing', prefix='small',
# )
# family_b_lgb = feature_ablation(
#     df_fe, candidate_features=CROSS_FAMILY_B,
#     model_name='LightGBM', include_all_combo=True,
# )

print('Family B 尚未实现；见上方 markdown 说明。')


Family B 尚未实现；见上方 markdown 说明。


## 6.3 OOF Autoencoder 重构误差

每折 **仅用正常样本** 训练 AE；折外 MSE 作为 `ae_oof_error`（越大越偏离正常流形）。
与树模型 **不同数学原理**，非树模型 `p_fraud`，无目标泄露。


In [6]:
from IPython.display import display

# =============================================================================
# 6.3 OOF Autoencoder — 计算 ae_oof_error 并 ablation
# -----------------------------------------------------------------------------
# ae_oof_error：与树模型不同原理的异常分，可作为 BASE 上的增量特征。
# 注意：全表 5-fold OOF 较慢（每折训一个 AE），首次运行需数分钟。
# =============================================================================

print('计算 OOF AE 重构误差（5-fold，可能需数分钟）…')
# 返回值长度 = len(df_fe)，按 index 对齐写入新列
df_fe['ae_oof_error'] = oof_ae_reconstruction_error(df_fe, feature_cols=AE_INPUT_COLS)
print(f"ae_oof_error: mean={df_fe['ae_oof_error'].mean():.6f}, "
      f"std={df_fe['ae_oof_error'].std():.6f}")

# --- 单特征 ablation：只加 ae_oof_error 一列 ---
# include_all_combo=False：候选只有 1 列时无需「+全部候选」行
print('\n=== OOF AE ablation（LightGBM）===')
ae_ablation_lgb = feature_ablation(
    df_fe, candidate_features=['ae_oof_error'],
    model_name='LightGBM', include_all_combo=False,
)
display(ae_ablation_lgb.round(4))

print('\n=== OOF AE ablation（XGBoost）===')
ae_ablation_xgb = feature_ablation(
    df_fe, candidate_features=['ae_oof_error'],
    model_name='XGBoost', include_all_combo=False,
)
display(ae_ablation_xgb.round(4))

# --- 可选：AE 误差 × 1 EUR 门控 ---
# 假设：异常分在 1 EUR 子空间更有判别力；乘 is_one_euro 后非 1 EUR 行为 0
# base_features 显式设为 BASE + ae_oof_error，测「在已有 AE 分上再加门控交叉」的边际收益
df_fe['ae_oof_error_x_one_euro'] = df_fe['ae_oof_error'] * df_fe['is_one_euro'].astype(float)
print('\n=== ae_oof_error × is_one_euro ablation（LightGBM）===')
ae_gate_ablation = feature_ablation(
    df_fe, base_features=BASE_FEATURES + ['ae_oof_error'],
    candidate_features=['ae_oof_error_x_one_euro'],
    model_name='LightGBM', include_all_combo=False,
)
display(ae_gate_ablation.round(4))


计算 OOF AE 重构误差（5-fold，可能需数分钟）…
  fold 1/5 完成（normal 训练样本 50,000）
  fold 2/5 完成（normal 训练样本 50,000）
  fold 3/5 完成（normal 训练样本 50,000）
  fold 4/5 完成（normal 训练样本 50,000）
  fold 5/5 完成（normal 训练样本 50,000）
ae_oof_error: mean=0.527973, std=3.485153

=== OOF AE ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8767,0.8469,0.8469,0.8469,15,15,0.0000
1,+ae_oof_error,31,0.8720,0.8557,0.8646,0.8469,13,15,-0.0047



=== OOF AE ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8756,0.8586,0.8817,0.8367,11,16,0.0000
1,+ae_oof_error,31,0.8742,0.8586,0.8817,0.8367,11,16,-0.0013



=== ae_oof_error × is_one_euro ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+ae_oof_error_x_one_euro,32,0.8787,0.8646,0.8830,0.8469,11,15,0.0067
1,基线,31,0.8720,0.8557,0.8646,0.8469,13,15,0.0000


## 6.4 组合验证 + 定稿 MODEL_FEATURES_V2

**6.4a**（5-fold CV，LGB / XGB **分表**）覆盖 **全部人为特征** 及组合：

| 块 | 内容 |
|----|------|
| **EDA 1.7** | `log1p_amount`、`hours_since_start`、`is_micro_testing`、`is_small_testing`、`is_one_euro`、`is_inactive`（单列 + 2.1d 递减组合 + curated） |
| **Family A** | `one_euro×Top-V`（单列 / Top-2/3 / 全族） |
| **OOF AE** | `ae_oof_error`、`ae_oof_error×is_one_euro` |
| **跨块** | EDA+AE、EDA+Family A、EDA+AE+Family A、**全部人为特征** |

**6.4b**：双模型 CV Δ 均为正且 Δ_mean 最高 → 定稿 `MODEL_FEATURES_V2`。


In [7]:
from IPython.display import display

# =============================================================================
# 6.4a 全部人为特征组合矩阵（EDA + Family A + OOF AE；5-fold CV；LGB/XGB 分表）
# =============================================================================

FE_EDA = list(EDA_FEATURES)  # 1.7：log1p_amount, hours, micro/small/one_euro, inactive
FE_AE = ['ae_oof_error']
FE_AE_GATE = ['ae_oof_error_x_one_euro']
FE_FAMILY_A = list(CROSS_FAMILY_A)
EDA_CURATED = ['hours_since_start', 'is_one_euro']  # 主 notebook 2.1d curated


def _dedupe_preserve_order(cols: list) -> list:
    seen, out = set(), []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def _extra_summary(extra_cols: list) -> str:
    if not extra_cols:
        return '—'
    s = ', '.join(extra_cols)
    return s if len(s) <= 72 else s[:69] + '...'


def _dedupe_specs(specs: list[tuple[list, str]]) -> list[tuple[list, str]]:
    seen: set[tuple[str, ...]] = set()
    out: list[tuple[list, str]] = []
    for extra, label in specs:
        key = tuple(_dedupe_preserve_order(extra))
        if key in seen:
            continue
        seen.add(key)
        out.append((list(key), label))
    return out


def build_fe_combo_specs(
    family_a_cols: list,
    eda_cols: list | None = None,
    fe_ae: list | None = None,
    fe_ae_gate: list | None = None,
    top_k_subsets: tuple[int, ...] = (2, 3),
) -> list[tuple[list, str]]:
    """
    构造 **全部人为特征** 的组合规格（相对 BASE_FEATURES 的增量列）。
    块：EDA 1.7 → OOF AE → Family A → 跨块并集。
    """
    eda_cols = list(eda_cols or FE_EDA)
    fe_ae = list(fe_ae or FE_AE)
    fe_ae_gate = list(fe_ae_gate or FE_AE_GATE)
    specs: list[tuple[list, str]] = []

    eda_minus_inactive = [c for c in eda_cols if c != 'is_inactive']
    eda_minus_inactive_small = [c for c in eda_cols if c not in ('is_inactive', 'is_small_testing')]
    eda_minus_no_hours = [c for c in eda_minus_inactive_small if c != 'hours_since_start']
    eda_minus_no_hours_log = [c for c in eda_minus_no_hours if c != 'log1p_amount']

    # --- 0. 基线 ---
    specs.append(([], '0. 基线（仅 BASE）'))

    # --- EDA：单列 ---
    for i, col in enumerate(eda_cols, start=1):
        specs.append(([col], f'Ed{i}. BASE + {col}'))

    # --- EDA：2.1d 式组合（镜像主 notebook）---
    specs.append((eda_cols, 'Ed_all. BASE + 全部 EDA（6 列）'))
    specs.append((eda_minus_inactive, 'Ed_noInact. BASE + EDA 去掉 is_inactive'))
    specs.append((eda_minus_inactive_small, 'Ed_noInactSm. BASE + EDA 去掉 inactive+small'))
    specs.append((eda_minus_no_hours, 'Ed_noHours. BASE + EDA 再去掉 hours'))
    specs.append((eda_minus_no_hours_log, 'Ed_noHoursLog. BASE + EDA 再去掉 log1p'))
    specs.append((EDA_CURATED, 'Ed_cur. BASE + hours + is_one_euro'))

    # --- OOF AE ---
    specs.append((fe_ae, 'AE. BASE + ae_oof_error'))
    specs.append((fe_ae + fe_ae_gate, 'AE+gate. BASE + ae_oof_error + ae×1EUR'))

    # --- EDA × AE ---
    specs.append((eda_cols + fe_ae, 'Ed_all+AE. 全部 EDA + ae_oof_error'))
    specs.append((EDA_CURATED + fe_ae, 'Ed_cur+AE. hours+is_one_euro + ae_oof_error'))
    specs.append((eda_minus_no_hours_log + fe_ae, 'Ed_noHoursLog+AE. EDA(无inactive/small/hours/log1p) + AE'))
    for col in eda_cols:
        specs.append((fe_ae + [col], f'AE+{col}. ae_oof_error + {col}'))

    # --- Family A：单列 / +AE / Top-k / 全族 ---
    for i, col in enumerate(family_a_cols, start=1):
        short = col.replace('one_euro_', '')
        specs.append(([col], f'A{i}. BASE + {col}（{short}）'))
        specs.append((fe_ae + [col], f'A{i}+AE. BASE + ae + {col}（{short}）'))
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        v_short = '+'.join(c.replace('one_euro_', '') for c in subset)
        specs.append((subset, f'A_top{k}. Family A Top-{k}（{v_short}）'))
        specs.append((fe_ae + subset, f'A_top{k}+AE. ae + Family A Top-{k}'))
    specs.append((family_a_cols, 'A_all. Family A 全族（6 列）'))
    specs.append((fe_ae + family_a_cols, 'A_all+AE. ae + Family A 全族'))

    # --- EDA × Family A（Top-k，不加 AE）---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + subset, f'Ed_all+A_top{k}. 全部 EDA + Family A Top-{k}'))
        specs.append((EDA_CURATED + subset, f'Ed_cur+A_top{k}. curated EDA + Family A Top-{k}'))

    # --- EDA × AE × Family A ---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + fe_ae + subset, f'Ed_all+AE+A_top{k}. 全部 EDA + ae + Family A Top-{k}'))
        specs.append((EDA_CURATED + fe_ae + subset, f'Ed_cur+AE+A_top{k}. curated + ae + Family A Top-{k}'))

    # --- 全量人为特征（EDA + AE + Family A + AE 门控）---
    all_handcrafted = _dedupe_preserve_order(eda_cols + fe_ae + family_a_cols + fe_ae_gate)
    specs.append((all_handcrafted, 'FULL. BASE + 全部人为特征（EDA+AE+FamilyA+gate）'))

    return _dedupe_specs(specs)


def eval_fe_combo(
    data: pd.DataFrame,
    extra_cols: list,
    label: str,
    model_name: str = 'LightGBM',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """5-fold CV 评估 BASE + extra_cols（比单次 hold-out 更稳）。"""
    extra_cols = _dedupe_preserve_order(list(extra_cols))
    feature_cols = _dedupe_preserve_order(BASE_FEATURES + extra_cols)
    missing = [c for c in feature_cols if c not in data.columns]
    if missing:
        raise KeyError(f'[{label}] 缺少列: {missing}')

    res = cross_val_eval(
        model_name, data, feature_cols,
        n_splits=n_splits, random_state=random_state, threshold=threshold,
    )
    return {
        '特征组合': label,
        '增量列数': len(extra_cols),
        '总特征数': len(feature_cols),
        '增量摘要': _extra_summary(extra_cols),
        '增量列': extra_cols,
        'AUC-PR_mean': res['AUC-PR_mean'],
        'AUC-PR_std': res['AUC-PR_std'],
        f'F1@{threshold}': res[f'F1@{threshold}'],
        f'Precision@{threshold}': res[f'Precision@{threshold}'],
        f'Recall@{threshold}': res[f'Recall@{threshold}'],
        'FP': res['FP'],
        'FN': res['FN'],
    }


def run_fe_combo_matrix(
    data: pd.DataFrame,
    combo_specs: list[tuple[list, str]],
    model_name: str = 'LightGBM',
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []
    n = len(combo_specs)
    for i, (extra, label) in enumerate(combo_specs, start=1):
        if verbose:
            print(f'  [{model_name}] CV {i}/{n}: {label}')
        rows.append(eval_fe_combo(data, extra, label, model_name=model_name))
    df_out = pd.DataFrame(rows)
    base_ap = float(df_out.loc[df_out['特征组合'].str.startswith('0.'), 'AUC-PR_mean'].iloc[0])
    df_out['Δ AUC-PR vs BASE'] = df_out['AUC-PR_mean'] - base_ap
    return df_out.sort_values('AUC-PR_mean', ascending=False).reset_index(drop=True)


def _display_combo_table(df: pd.DataFrame, title: str) -> None:
    """展示组合表（隐藏增量列 list，用增量摘要代替）。"""
    show_cols = [c for c in df.columns if c != '增量列']
    print(title)
    display(df[show_cols].round(4))


COMBO_SPECS = build_fe_combo_specs(FE_FAMILY_A)
_n_eda = len(FE_EDA)
print(f'共 {len(COMBO_SPECS)} 种组合 × {CV_N_SPLITS}-fold CV')
print(f'  含 EDA 1.7（{_n_eda} 列单列+组合）、Family A、OOF AE、跨块 FULL 等\n')

print('=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_lgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='LightGBM')
_display_combo_table(combo_lgb, '')

print(f'\n=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_xgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='XGBoost')
_display_combo_table(combo_xgb, '')

print('\n--- LightGBM：CV Δ>0 的前 3 名 ---')
display(combo_lgb.loc[combo_lgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

print('--- XGBoost：CV Δ>0 的前 3 名 ---')
display(combo_xgb.loc[combo_xgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

# 供 6.4b 定稿：内部合并，不在此 cell 展示宽表
combo_lgb_idx = combo_lgb.set_index('特征组合')
combo_xgb_idx = combo_xgb.set_index('特征组合')
_common_labels = combo_lgb_idx.index.intersection(combo_xgb_idx.index)
combo_dual_rows = []
for label in _common_labels:
    dl = float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    dx = float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    combo_dual_rows.append({
        '特征组合': label,
        '增量列': combo_lgb_idx.loc[label, '增量列'],
        'Δ_LGB': dl,
        'Δ_XGB': dx,
        'Δ_mean': (dl + dx) / 2,
        '双模型Δ均为正': dl > 0 and dx > 0,
    })
combo_dual = pd.DataFrame(combo_dual_rows).sort_values('Δ_mean', ascending=False).reset_index(drop=True)

共 51 种组合 × 5-fold CV
  含 EDA 1.7（6 列单列+组合）、Family A、OOF AE、跨块 FULL 等

=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [LightGBM] CV 1/51: 0. 基线（仅 BASE）
  [LightGBM] CV 2/51: Ed1. BASE + log1p_amount
  [LightGBM] CV 3/51: Ed2. BASE + hours_since_start
  [LightGBM] CV 4/51: Ed3. BASE + is_micro_testing
  [LightGBM] CV 5/51: Ed4. BASE + is_small_testing
  [LightGBM] CV 6/51: Ed5. BASE + is_one_euro
  [LightGBM] CV 7/51: Ed6. BASE + is_inactive
  [LightGBM] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [LightGBM] CV 9/51: Ed_noInact. BASE + EDA 去掉 is_inactive
  [LightGBM] CV 10/51: Ed_noInactSm. BASE + EDA 去掉 inactive+small
  [LightGBM] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [LightGBM] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [LightGBM] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [LightGBM] CV 14/51: AE. BASE + ae_oof_error
  [LightGBM] CV 15/51: AE+gate. BASE + ae_oof_error + ae×1EUR
  [LightGBM] CV 16/51: Ed_all+AE. 全部 EDA + ae_oof_error
  [LightGBM] CV 17/51: Ed_cur+AE. hours+i

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_cur+AE. hours+is_one_euro + ae_oof_error,3,33,"hours_since_start, is_one_euro, ae_oof_error",0.8617,0.0214,0.8672,0.9087,0.8293,41,84,0.0040
1,Ed_all+AE. 全部 EDA + ae_oof_error,7,37,"log1p_amount, hours_since_start, is_micro_test...",0.8616,0.0227,0.8662,0.9067,0.8293,42,84,0.0038
2,A_all+AE. ae + Family A 全族,7,37,"ae_oof_error, one_euro_V14, one_euro_V4, one_e...",0.8614,0.0210,0.8657,0.9103,0.8252,40,86,0.0037
3,A4+AE. BASE + ae + one_euro_V16（V16）,2,32,"ae_oof_error, one_euro_V16",0.8614,0.0215,0.8647,0.9009,0.8313,45,83,0.0036
4,A6+AE. BASE + ae + one_euro_V26（V26）,2,32,"ae_oof_error, one_euro_V26",0.8613,0.0197,0.8620,0.9097,0.8191,40,89,0.0035
5,AE+log1p_amount. ae_oof_error + log1p_amount,2,32,"ae_oof_error, log1p_amount",0.8611,0.0212,0.8617,0.9040,0.8232,43,87,0.0033
6,A_top2+AE. ae + Family A Top-2,3,33,"ae_oof_error, one_euro_V14, one_euro_V4",0.8608,0.0225,0.8684,0.9089,0.8313,41,83,0.0031
7,AE. BASE + ae_oof_error,1,31,ae_oof_error,0.8605,0.0228,0.8629,0.9042,0.8252,43,86,0.0028
8,AE+hours_since_start. ae_oof_error + hours_sin...,2,32,"ae_oof_error, hours_since_start",0.8603,0.0219,0.8611,0.9002,0.8252,45,86,0.0026
9,Ed_noInactSm. BASE + EDA 去掉 inactive+small,4,34,"log1p_amount, hours_since_start, is_micro_test...",0.8603,0.0226,0.8699,0.9148,0.8293,38,84,0.0026



=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [XGBoost] CV 1/51: 0. 基线（仅 BASE）
  [XGBoost] CV 2/51: Ed1. BASE + log1p_amount
  [XGBoost] CV 3/51: Ed2. BASE + hours_since_start
  [XGBoost] CV 4/51: Ed3. BASE + is_micro_testing
  [XGBoost] CV 5/51: Ed4. BASE + is_small_testing
  [XGBoost] CV 6/51: Ed5. BASE + is_one_euro
  [XGBoost] CV 7/51: Ed6. BASE + is_inactive
  [XGBoost] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [XGBoost] CV 9/51: Ed_noInact. BASE + EDA 去掉 is_inactive
  [XGBoost] CV 10/51: Ed_noInactSm. BASE + EDA 去掉 inactive+small
  [XGBoost] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [XGBoost] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [XGBoost] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [XGBoost] CV 14/51: AE. BASE + ae_oof_error
  [XGBoost] CV 15/51: AE+gate. BASE + ae_oof_error + ae×1EUR
  [XGBoost] CV 16/51: Ed_all+AE. 全部 EDA + ae_oof_error
  [XGBoost] CV 17/51: Ed_cur+AE. hours+is_one_euro + ae_oof_error
  [XGBoost] CV 18/51: Ed_noHoursLog+AE. EDA(无inactive/small/h

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,AE+is_micro_testing. ae_oof_error + is_micro_t...,2,32,"ae_oof_error, is_micro_testing",0.8634,0.0243,0.8656,0.9029,0.8313,44,83,0.0056
1,A1+AE. BASE + ae + one_euro_V14（V14）,2,32,"ae_oof_error, one_euro_V14",0.8627,0.0250,0.8599,0.8928,0.8293,49,84,0.0049
2,Ed_cur+AE+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, ae_oof_error, ...",0.8626,0.0238,0.8599,0.8928,0.8293,49,84,0.0048
3,AE+gate. BASE + ae_oof_error + ae×1EUR,2,32,"ae_oof_error, ae_oof_error_x_one_euro",0.8619,0.0239,0.8589,0.8908,0.8293,50,84,0.0041
4,FULL. BASE + 全部人为特征（EDA+AE+FamilyA+gate）,14,44,"log1p_amount, hours_since_start, is_micro_test...",0.8617,0.0248,0.8574,0.8923,0.8252,49,86,0.0039
5,Ed_all+A_top3. 全部 EDA + Family A Top-3,9,39,"log1p_amount, hours_since_start, is_micro_test...",0.8616,0.0238,0.8638,0.8989,0.8313,46,83,0.0038
6,Ed_all+AE+A_top2. 全部 EDA + ae + Family A Top-2,9,39,"log1p_amount, hours_since_start, is_micro_test...",0.8615,0.0255,0.8620,0.8950,0.8313,48,83,0.0037
7,Ed_cur+A_top3. curated EDA + Family A Top-3,5,35,"hours_since_start, is_one_euro, one_euro_V14, ...",0.8615,0.0260,0.8638,0.8989,0.8313,46,83,0.0037
8,A_top3. Family A Top-3（V14+V4+V7）,3,33,"one_euro_V14, one_euro_V4, one_euro_V7",0.8613,0.0269,0.8611,0.8930,0.8313,49,83,0.0035
9,AE+is_inactive. ae_oof_error + is_inactive,2,32,"ae_oof_error, is_inactive",0.8612,0.0265,0.8608,0.8947,0.8293,48,84,0.0034



--- LightGBM：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_cur+AE. hours+is_one_euro + ae_oof_error,3,33,"hours_since_start, is_one_euro, ae_oof_error",0.8617,0.0214,0.8672,0.9087,0.8293,41,84,0.0040
1,Ed_all+AE. 全部 EDA + ae_oof_error,7,37,"log1p_amount, hours_since_start, is_micro_test...",0.8616,0.0227,0.8662,0.9067,0.8293,42,84,0.0038
2,A_all+AE. ae + Family A 全族,7,37,"ae_oof_error, one_euro_V14, one_euro_V4, one_e...",0.8614,0.0210,0.8657,0.9103,0.8252,40,86,0.0037


--- XGBoost：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,AE+is_micro_testing. ae_oof_error + is_micro_t...,2,32,"ae_oof_error, is_micro_testing",0.8634,0.0243,0.8656,0.9029,0.8313,44,83,0.0056
1,A1+AE. BASE + ae + one_euro_V14（V14）,2,32,"ae_oof_error, one_euro_V14",0.8627,0.0250,0.8599,0.8928,0.8293,49,84,0.0049
2,Ed_cur+AE+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, ae_oof_error, ...",0.8626,0.0238,0.8599,0.8928,0.8293,49,84,0.0048


考虑用FamilyA Top3+is_one_euro+hours_since_start。因为波动有波动，且之前某次调整参数发现Top3对两个模型几乎都是最优（不过那次没有hours和oneeuro）

In [8]:
# =============================================================================
# 6.4b 定稿 MODEL_FEATURES_V2
# -----------------------------------------------------------------------------
# 展示：LGB / XGB 各自最优（Δ>0）；定稿：双模型 Δ 均为正且 Δ_mean 最高（combo_dual）
# =============================================================================

def _ablation_combo_delta(ablation_df: pd.DataFrame) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == '+全部候选']
    if row.empty:
        row = ablation_df.iloc[[0]]
    return float(row['Δ AUC-PR'].iloc[0])


def _ablation_single_delta(ablation_df: pd.DataFrame, tag: str) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == tag]
    return float(row['Δ AUC-PR'].iloc[0]) if not row.empty else 0.0


def _best_positive_row(combo_df: pd.DataFrame) -> pd.Series | None:
    pos = combo_df.loc[combo_df['Δ AUC-PR vs BASE'] > 0]
    return pos.iloc[0] if not pos.empty else None


decisions = []

# --- 各模型单独最优（仅展示，不一定作最终定稿）---
best_lgb = _best_positive_row(combo_lgb)
best_xgb = _best_positive_row(combo_xgb)

print('=== 6.4b LightGBM 单模型最优（CV Δ>0）===')
if best_lgb is not None:
    print(f"  {best_lgb['特征组合']}  |  Δ={best_lgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_lgb['AUC-PR_mean']:.4f}±{best_lgb['AUC-PR_std']:.4f}  |  {best_lgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

print('\n=== 6.4b XGBoost 单模型最优（CV Δ>0）===')
if best_xgb is not None:
    print(f"  {best_xgb['特征组合']}  |  Δ={best_xgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_xgb['AUC-PR_mean']:.4f}±{best_xgb['AUC-PR_std']:.4f}  |  {best_xgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

# --- 双模型一致定稿 ---
eligible = combo_dual[combo_dual['双模型Δ均为正']].sort_values('Δ_mean', ascending=False)
if eligible.empty:
    WINNER_LABEL = '0. 基线（仅 BASE）'
    SELECTED_EXTRA = []
    decisions.append('组合定稿：无「双模型 CV Δ 均为正」方案 → 保留 BASE_FEATURES')
else:
    winner = eligible.iloc[0]
    WINNER_LABEL = winner['特征组合']
    SELECTED_EXTRA = list(winner['增量列'])
    decisions.append(
        f"组合定稿：{WINNER_LABEL} "
        f"(LGB Δ={winner['Δ_LGB']:.4f}, XGB Δ={winner['Δ_XGB']:.4f}, Δ_mean={winner['Δ_mean']:.4f})"
    )

MODEL_FEATURES_V2 = BASE_FEATURES + [c for c in SELECTED_EXTRA if c not in BASE_FEATURES]

decisions.append(
    f'[参考] Family A 单族 ablation: LGB Δ={_ablation_combo_delta(family_a_lgb):.4f}, '
    f'XGB Δ={_ablation_combo_delta(family_a_xgb):.4f}'
)
decisions.append(
    f'[参考] ae_oof_error 单列 ablation: LGB Δ={_ablation_single_delta(ae_ablation_lgb, "+ae_oof_error"):.4f}, '
    f'XGB Δ={_ablation_single_delta(ae_ablation_xgb, "+ae_oof_error"):.4f}'
)

print('\n=== 6.4b 定稿决策 ===')
for d in decisions:
    print(' -', d)
print(f'\nMODEL_FEATURES_V2（{len(MODEL_FEATURES_V2)} 列，相对 BASE 增量 {len(MODEL_FEATURES_V2) - len(BASE_FEATURES)}）：')
print(MODEL_FEATURES_V2)

print('\n提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。')

import json as _json
export_path = 'MODEL_FEATURES_V2.json'
with open(export_path, 'w', encoding='utf-8') as f:
    _json.dump(
        {
            'MODEL_FEATURES_V2': MODEL_FEATURES_V2,
            'winner_combo': WINNER_LABEL,
            'selected_extra': SELECTED_EXTRA,
            'best_lgb_combo': None if best_lgb is None else best_lgb['特征组合'],
            'best_xgb_combo': None if best_xgb is None else best_xgb['特征组合'],
            'cv_n_splits': CV_N_SPLITS,
            'decisions': decisions,
            'combo_lgb': combo_lgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
            'combo_xgb': combo_xgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
        },
        f, ensure_ascii=False, indent=2,
    )
print(f'已导出 → {export_path}')


=== 6.4b LightGBM 单模型最优（CV Δ>0）===
  Ed_cur+AE. hours+is_one_euro + ae_oof_error  |  Δ=0.0040  |  AUC-PR=0.8617±0.0214  |  hours_since_start, is_one_euro, ae_oof_error

=== 6.4b XGBoost 单模型最优（CV Δ>0）===
  AE+is_micro_testing. ae_oof_error + is_micro_testing  |  Δ=0.0056  |  AUC-PR=0.8634±0.0243  |  ae_oof_error, is_micro_testing

=== 6.4b 定稿决策 ===
 - 组合定稿：Ed_cur+AE+A_top2. curated + ae + Family A Top-2 (LGB Δ=0.0023, XGB Δ=0.0048, Δ_mean=0.0036)
 - [参考] Family A 单族 ablation: LGB Δ=0.0008, XGB Δ=-0.0052
 - [参考] ae_oof_error 单列 ablation: LGB Δ=-0.0047, XGB Δ=-0.0013

MODEL_FEATURES_V2（35 列，相对 BASE 增量 5）：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time', 'hours_since_start', 'is_one_euro', 'ae_oof_error', 'one_euro_V14', 'one_euro_V4']

提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。
已导出 → MODEL_FEATURES_V2.json
